In [ ]:
# In[1]:

import json
from datetime import datetime
import pytz

# Define the task directory
task_directory = 'rca100/cases/t069'

# Read the task.json file
with open(f'{task_directory}/task.json', 'r') as file:
    task_data = json.load(file)

# Extract alert context
alert_title = task_data.get('alert_title', None)
alert_window = task_data.get('alert_window', {})
alert_start = datetime.fromisoformat(alert_window.get('start')).astimezone(pytz.timezone('Asia/Shanghai'))
alert_end = datetime.fromisoformat(alert_window.get('end')).astimezone(pytz.timezone('Asia/Shanghai'))
alert_entity = task_data.get('alert_entity', {})

alert_title, alert_start, alert_end, alert_entity

```
Out[1]:
```
The alert is titled "product-catalog 流量下跌告警" (product-catalog traffic drop alert). It occurred between 2026-04-13 04:59:52 and 2026-04-13 05:05:32 (UTC+8). The entity that triggered the alert is the operation `product-catalog::oteldemo.ProductCatalogService/ListProducts` of type `apm.operation` in the `apm` domain.

The original code execution output of IPython Kernel is also provided below for reference:

('product-catalog 流量下跌告警', datetime.datetime(2026, 4, 13, 4, 59, 52, tzinfo=<DstTzInfo 'Asia/Shanghai' CST+8:00:00 STD>), datetime.datetime(2026, 4, 13, 5, 5, 32, tzinfo=<DstTzInfo 'Asia/Shanghai' CST+8:00:00 STD>), {'entity_id': '455f921f0afc16175724d9f27faf18e4', 'entity_name': 'product-catalog::oteldemo.ProductCatalogService/ListProducts', 'entity_type': 'apm.operation', 'entity_domain': 'apm'})```
```

In [ ]:
# In[2]:

import pandas as pd

# Load the metrics.parquet file
metrics_file = f'{task_directory}/metrics.parquet'
metrics_data = pd.read_parquet(metrics_file)

# Display all available KPIs (unique metric names)
available_kpis = metrics_data['metric'].unique()
available_kpis

```
Out[2]:
```
The available KPIs in the metrics data include traffic-related metrics such as `request_count`, `error_count`, `error_rate`, and `latency`. These metrics are relevant for analyzing the traffic drop issue. Other metrics like `slow`, `slow_count`, and various deployment, node, and JVM-related metrics are also available for further analysis if needed.

The original code execution output of IPython Kernel is also provided below for reference:

['node_ready_status' 'node_not_ready_status' 'node_memory_usage_rate'
 'node_disk_usage_rate' 'node_pod_running_count' 'node_cpu_usage_rate'
 'deployment_available_replicas' 'deployment_desired_replicas'
 'deployment_ready_replicas' 'deployment_availability_rate'
 'deployment_memory_usage_total' 'deployment_cpu_usage_vs_limits'
 'deployment_cpu_usage_total' 'deployment_cpu_usage_vs_requests'
 'cluster_node_count' 'cluster_failed_node_count'
 'apiserver_non_2xx_read_request_qps' 'cluster_number_of_not_ready_pods'
 'apiserver_non_2xx_write_request_qps' 'apiserver_concurrent_connections'
 'cluster_number_of_pod' 'apiserver_requests_per_second'
 'namespace_number_of_not_ready_pods' 'namespace_number_of_pods'
 'slow_count' 'error_count' 'request_count' 'avg_request_latency_seconds'
 'arms_jvm_gc_total' 'arms_jvm_gc_seconds_total' 'arms_jvm_gc_delta'
 'arms_jvm_gc_seconds_delta' 'arms_jvm_mem_used_bytes'
 'arms_jvm_mem_init_bytes' 'arms_jvm_mem_committed_bytes'
 'arms_jvm_mem_max_bytes' 'arms_jvm_threads_count'
 'arms_thread_pool_active_thread_count'
 'arms_thread_pool_current_thread_count'
 'arms_thread_pool_max_thread_count' 'arms_thread_pool_max_pool_size'
 'arms_thread_pool_core_pool_size' 'arms_thread_pool_scheduled_task_count'
 'arms_thread_pool_completed_task_count'
 'arms_thread_pool_rejected_task_count' 'arms_thread_pool_queue_size'
 'arms_grouped_thread_count' 'arms_grouped_thread_cpu_time'
 'arms_grouped_thread_runnable_count' 'arms_grouped_thread_waiting_count'
 'arms_grouped_thread_timed_waiting_count' 'cpu' 'latency' 'mem'
 'workload' 'slow' 'error' 'error_rate']```
```

In [ ]:
# In[3]:

# Filter the metrics data for the `request_count` metric and the `product-catalog` service
metrics_data['time'] = pd.to_numeric(metrics_data['time'], errors='coerce')  # Ensure time is numeric
metrics_data['value'] = pd.to_numeric(metrics_data['value'], errors='coerce')  # Ensure value is numeric

product_catalog_request_count = metrics_data[
    (metrics_data['metric'] == 'request_count') &
    (metrics_data['entity_name'].str.contains('product-catalog', case=False))
]

# Calculate the global P95 threshold for the `request_count` metric
global_p95_threshold = product_catalog_request_count['value'].quantile(0.95)

# Filter the data for the incident time window
incident_start = alert_start.timestamp() * 1_000_000  # Convert to microseconds
incident_end = alert_end.timestamp() * 1_000_000  # Convert to microseconds

incident_data = product_catalog_request_count[
    (product_catalog_request_count['time'] >= incident_start) &
    (product_catalog_request_count['time'] <= incident_end)
]

global_p95_threshold, incident_data

```
Out[3]:
```
The global P95 threshold for the `request_count` metric across the entire dataset is 10,587.25. During the incident time window (2026-04-13 04:59:52 to 2026-04-13 05:05:32, UTC+8), the `request_count` metric for the `product-catalog` service shows significantly lower values, ranging from 1,370 to 12,849 across different entities. This indicates a substantial drop in traffic compared to the global P95 threshold.

The original code execution output of IPython Kernel is also provided below for reference:

(np.float64(10587.25),                    time domain     entity_set                         entity_id                                        entity_name         metric    value                            metric_set_id service
57359  1776027592000000    apm  apm.operation  455f921f0afc16175724d9f27faf18e4  product-catalog::oteldemo.ProductCatalogServic...  request_count   5600.0  apm@metric_set@apm.metric.apm.operation    None
57360  1776027622000000    apm  apm.operation  455f921f0afc16175724d9f27faf18e4  product-catalog::oteldemo.ProductCatalogServic...  request_count   3818.0  apm@metric_set@apm.metric.apm.operation    None
57361  1776027652000000    apm  apm.operation  455f921f0afc16175724d9f27faf18e4  product-catalog::oteldemo.ProductCatalogServic...  request_count   1370.0  apm@metric_set@apm.metric.apm.operation    None
57362  1776027682000000    apm  apm.operation  455f921f0afc16175724d9f27faf18e4  product-catalog::oteldemo.ProductCatalogServic...  request_count   1469.0  apm@metric_set@apm.metric.apm.operation    None
57363  1776027712000000    apm  apm.operation  455f921f0afc16175724d9f27faf18e4  product-catalog::oteldemo.ProductCatalogServic...  request_count   1743.0  apm@metric_set@apm.metric.apm.operation    None
57364  1776027742000000    apm  apm.operation  455f921f0afc16175724d9f27faf18e4  product-catalog::oteldemo.ProductCatalogServic...  request_count   1745.0  apm@metric_set@apm.metric.apm.operation    None
57365  1776027772000000    apm  apm.operation  455f921f0afc16175724d9f27faf18e4  product-catalog::oteldemo.ProductCatalogServic...  request_count   1723.0  apm@metric_set@apm.metric.apm.operation    None
57366  1776027802000000    apm  apm.operation  455f921f0afc16175724d9f27faf18e4  product-catalog::oteldemo.ProductCatalogServic...  request_count   1601.0  apm@metric_set@apm.metric.apm.operation    None
57367  1776027832000000    apm  apm.operation  455f921f0afc16175724d9f27faf18e4  product-catalog::oteldemo.ProductCatalogServic...  request_count   1502.0  apm@metric_set@apm.metric.apm.operation    None
57368  1776027862000000    apm  apm.operation  455f921f0afc16175724d9f27faf18e4  product-catalog::oteldemo.ProductCatalogServic...  request_count   1582.0  apm@metric_set@apm.metric.apm.operation    None
57369  1776027892000000    apm  apm.operation  455f921f0afc16175724d9f27faf18e4  product-catalog::oteldemo.ProductCatalogServic...  request_count   1652.0  apm@metric_set@apm.metric.apm.operation    None
57370  1776027922000000    apm  apm.operation  455f921f0afc16175724d9f27faf18e4  product-catalog::oteldemo.ProductCatalogServic...  request_count   1612.0  apm@metric_set@apm.metric.apm.operation    None
60184  1776027592000000    apm  apm.operation  1948ddf615871ac6fc18d01f72817e86  product-catalog::oteldemo.ProductCatalogServic...  request_count  12849.0  apm@metric_set@apm.metric.apm.operation    None
60185  1776027622000000    apm  apm.operation  1948ddf615871ac6fc18d01f72817e86  product-catalog::oteldemo.ProductCatalogServic...  request_count   9209.0  apm@metric_set@apm.metric.apm.operation    None
60186  1776027652000000    apm  apm.operation  1948ddf615871ac6fc18d01f72817e86  product-catalog::oteldemo.ProductCatalogServic...  request_count   3492.0  apm@metric_set@apm.metric.apm.operation    None
60187  1776027682000000    apm  apm.operation  1948ddf615871ac6fc18d01f72817e86  product-catalog::oteldemo.ProductCatalogServic...  request_count   3414.0  apm@metric_set@apm.metric.apm.operation    None
60188  1776027712000000    apm  apm.operation  1948ddf615871ac6fc18d01f72817e86  product-catalog::oteldemo.ProductCatalogServic...  request_count   3822.0  apm@metric_set@apm.metric.apm.operation    None
60189  1776027742000000    apm  apm.operation  1948ddf615871ac6fc18d01f72817e86  product-catalog::oteldemo.ProductCatalogServic...  request_count   3796.0  apm@metric_set@apm.metric.apm.operation    None
60190  1776027772000000    apm  apm.operation  1948ddf615871ac6fc18d01f72817e86  product-catalog::oteldemo.ProductCatalogServic...  request_count   3818.0  apm@metric_set@apm.metric.apm.operation    None
60191  1776027802000000    apm  apm.operation  1948ddf615871ac6fc18d01f72817e86  product-catalog::oteldemo.ProductCatalogServic...  request_count   3493.0  apm@metric_set@apm.metric.apm.operation    None
60192  1776027832000000    apm  apm.operation  1948ddf615871ac6fc18d01f72817e86  product-catalog::oteldemo.ProductCatalogServic...  request_count   3281.0  apm@metric_set@apm.metric.apm.operation    None
60193  1776027862000000    apm  apm.operation  1948ddf615871ac6fc18d01f72817e86  product-catalog::oteldemo.ProductCatalogServic...  request_count   3607.0  apm@metric_set@apm.metric.apm.operation    None
60194  1776027892000000    apm  apm.operation  1948ddf615871ac6fc18d01f72817e86  product-catalog::oteldemo.ProductCatalogServic...  request_count   3733.0  apm@metric_set@apm.metric.apm.operation    None
60195  1776027922000000    apm  apm.operation  1948ddf615871ac6fc18d01f72817e86  product-catalog::oteldemo.ProductCatalogServic...  request_count   3467.0  apm@metric_set@apm.metric.apm.operation    None
68603  1776027592000000    apm   apm.instance  df6348543e3360ea1206bd20179899fc                    product-catalog-79f7b8c4b-rmpgh  request_count  12415.0   apm@metric_set@apm.metric.apm.instance    None
68604  1776027622000000    apm   apm.instance  df6348543e3360ea1206bd20179899fc                    product-catalog-79f7b8c4b-rmpgh  request_count   8442.0   apm@metric_set@apm.metric.apm.instance    None
68605  1776027652000000    apm   apm.instance  df6348543e3360ea1206bd20179899fc                    product-catalog-79f7b8c4b-rmpgh  request_count   2508.0   apm@metric_set@apm.metric.apm.instance    None
68606  1776027682000000    apm   apm.instance  df6348543e3360ea1206bd20179899fc                    product-catalog-79f7b8c4b-rmpgh  request_count   2552.0   apm@metric_set@apm.metric.apm.instance    None
68607  1776027712000000    apm   apm.instance  df6348543e3360ea1206bd20179899fc                    product-catalog-79f7b8c4b-rmpgh  request_count   2870.0   apm@metric_set@apm.metric.apm.instance    None
68608  1776027742000000    apm   apm.instance  df6348543e3360ea1206bd20179899fc                    product-catalog-79f7b8c4b-rmpgh  request_count   2867.0   apm@metric_set@apm.metric.apm.instance    None
68609  1776027772000000    apm   apm.instance  df6348543e3360ea1206bd20179899fc                    product-catalog-79f7b8c4b-rmpgh  request_count   3252.0   apm@metric_set@apm.metric.apm.instance    None
68610  1776027802000000    apm   apm.instance  df6348543e3360ea1206bd20179899fc                    product-catalog-79f7b8c4b-rmpgh  request_count   3225.0   apm@metric_set@apm.metric.apm.instance    None
68611  1776027832000000    apm   apm.instance  df6348543e3360ea1206bd20179899fc                    product-catalog-79f7b8c4b-rmpgh  request_count   2923.0   apm@metric_set@apm.metric.apm.instance    None
68612  1776027862000000    apm   apm.instance  df6348543e3360ea1206bd20179899fc                    product-catalog-79f7b8c4b-rmpgh  request_count   3181.0   apm@metric_set@apm.metric.apm.instance    None
68613  1776027892000000    apm   apm.instance  df6348543e3360ea1206bd20179899fc                    product-catalog-79f7b8c4b-rmpgh  request_count   3409.0   apm@metric_set@apm.metric.apm.instance    None
68614  1776027922000000    apm   apm.instance  df6348543e3360ea1206bd20179899fc                    product-catalog-79f7b8c4b-rmpgh  request_count   3132.0   apm@metric_set@apm.metric.apm.instance    None
68739  1776027592000000    apm   apm.instance  34e3e1c865aacce7dbda259b7ca3cefc                    product-catalog-79f7b8c4b-lmp9q  request_count   6034.0   apm@metric_set@apm.metric.apm.instance    None
68740  1776027622000000    apm   apm.instance  34e3e1c865aacce7dbda259b7ca3cefc                    product-catalog-79f7b8c4b-lmp9q  request_count   4585.0   apm@metric_set@apm.metric.apm.instance    None
68741  1776027652000000    apm   apm.instance  34e3e1c865aacce7dbda259b7ca3cefc                    product-catalog-79f7b8c4b-lmp9q  request_count   2354.0   apm@metric_set@apm.metric.apm.instance    None
68742  1776027682000000    apm   apm.instance  34e3e1c865aacce7dbda259b7ca3cefc                    product-catalog-79f7b8c4b-lmp9q  request_count   2331.0   apm@metric_set@apm.metric.apm.instance    None
68743  1776027712000000    apm   apm.instance  34e3e1c865aacce7dbda259b7ca3cefc                    product-catalog-79f7b8c4b-lmp9q  request_count   2695.0   apm@metric_set@apm.metric.apm.instance    None
68744  1776027742000000    apm   apm.instance  34e3e1c865aacce7dbda259b7ca3cefc                    product-catalog-79f7b8c4b-lmp9q  request_count   2674.0   apm@metric_set@apm.metric.apm.instance    None
68745  1776027772000000    apm   apm.instance  34e3e1c865aacce7dbda259b7ca3cefc                    product-catalog-79f7b8c4b-lmp9q  request_count   2289.0   apm@metric_set@apm.metric.apm.instance    None
68746  1776027802000000    apm   apm.instance  34e3e1c865aacce7dbda259b7ca3cefc                    product-catalog-79f7b8c4b-lmp9q  request_count   1869.0   apm@metric_set@apm.metric.apm.instance    None
68747  1776027832000000    apm   apm.instance  34e3e1c865aacce7dbda259b7ca3cefc                    product-catalog-79f7b8c4b-lmp9q  request_count   1860.0   apm@metric_set@apm.metric.apm.instance    None
68748  1776027862000000    apm   apm.instance  34e3e1c865aacce7dbda259b7ca3cefc                    product-catalog-79f7b8c4b-lmp9q  request_count   2008.0   apm@metric_set@apm.metric.apm.instance    None
68749  1776027892000000    apm   apm.instance  34e3e1c865aacce7dbda259b7ca3cefc                    product-catalog-79f7b8c4b-lmp9q  request_count   1976.0   apm@metric_set@apm.metric.apm.instance    None
68750  1776027922000000    apm   apm.instance  34e3e1c865aacce7dbda259b7ca3cefc                    product-catalog-79f7b8c4b-lmp9q  request_count   1947.0   apm@metric_set@apm.metric.apm.instance    None)```
```

In [ ]:
# In[4]:

# Filter the metrics data for the `error_rate` metric and the `product-catalog` service
product_catalog_error_rate = metrics_data[
    (metrics_data['metric'] == 'error_rate') &
    (metrics_data['entity_name'].str.contains('product-catalog', case=False))
]

# Calculate the global P95 threshold for the `error_rate` metric
error_rate_global_p95_threshold = product_catalog_error_rate['value'].quantile(0.95)

# Filter the data for the incident time window
error_rate_incident_data = product_catalog_error_rate[
    (product_catalog_error_rate['time'] >= incident_start) &
    (product_catalog_error_rate['time'] <= incident_end)
]

error_rate_global_p95_threshold, error_rate_incident_data

```
Out[4]:
```
The `error_rate` metric for the `product-catalog` service does not have any data available in the dataset. As a result, the global P95 threshold could not be calculated, and no data is available for the incident time window.

The original code execution output of IPython Kernel is also provided below for reference:

(np.float64(nan), Empty DataFrame
Columns: [time, domain, entity_set, entity_id, entity_name, metric, value, metric_set_id, service]
Index: [])```
```

In [ ]:
# In[5]:

# Load the traces.parquet file
traces_file = f'{task_directory}/traces.parquet'
traces_data = pd.read_parquet(traces_file)

# Ensure relevant columns are numeric
traces_data['startTime'] = pd.to_numeric(traces_data['startTime'], errors='coerce')
traces_data['endTime'] = pd.to_numeric(traces_data['endTime'], errors='coerce')
traces_data['duration'] = pd.to_numeric(traces_data['duration'], errors='coerce')

# Filter traces for the `product-catalog::oteldemo.ProductCatalogService/ListProducts` operation during the incident time window
filtered_traces = traces_data[
    (traces_data['spanName'] == 'product-catalog::oteldemo.ProductCatalogService/ListProducts') &
    (traces_data['startTime'] >= incident_start * 1_000) &  # Convert to nanoseconds
    (traces_data['endTime'] <= incident_end * 1_000)        # Convert to nanoseconds
]

# Identify downstream services or operations with errors or high latency
downstream_traces = traces_data[
    traces_data['parentSpanId'].isin(filtered_traces['spanId'])
]

# Analyze for errors or high latency
downstream_traces_with_issues = downstream_traces[
    (downstream_traces['statusCode'] != 0) |  # Non-zero status code indicates an error
    (downstream_traces['duration'] > downstream_traces['duration'].quantile(0.95))  # High latency (above P95)
]

filtered_traces, downstream_traces_with_issues

```
Out[5]:
```
No trace data is available for the `product-catalog::oteldemo.ProductCatalogService/ListProducts` operation during the incident time window. Consequently, no downstream services or operations with errors or high latency could be identified.

The original code execution output of IPython Kernel is also provided below for reference:

(Empty DataFrame
Columns: [traceId, spanId, parentSpanId, kind, spanName, startTime, endTime, duration, serviceName, pid, ip, statusCode, statusMessage, traceState, resources, attributes, __topic__, __tag__:__pack_id__, hostname, events, links]
Index: []

[0 rows x 21 columns], Empty DataFrame
Columns: [traceId, spanId, parentSpanId, kind, spanName, startTime, endTime, duration, serviceName, pid, ip, statusCode, statusMessage, traceState, resources, attributes, __topic__, __tag__:__pack_id__, hostname, events, links]
Index: []

[0 rows x 21 columns])```
```

In [ ]:
# In[6]:

# Load the logs.parquet file
logs_file = f'{task_directory}/logs.parquet'
logs_data = pd.read_parquet(logs_file)

# Ensure the `_time_` column is in numeric format
logs_data['_time_'] = pd.to_numeric(logs_data['_time_'], errors='coerce')

# Filter logs for the `product-catalog` service during the incident time window
product_catalog_logs = logs_data[
    (logs_data['_time_'] >= incident_start) &
    (logs_data['_time_'] <= incident_end) &
    (logs_data['_container_name_'].str.contains('product-catalog', case=False, na=False))
]

# Search for error messages or anomalies in the log content
error_logs = product_catalog_logs[
    product_catalog_logs['content'].str.contains('error|exception|fail|critical', case=False, na=False)
]

product_catalog_logs, error_logs

```
Out[6]:
```
No logs are available for the `product-catalog` service during the incident time window. As a result, no error messages or anomalies could be identified in the log content.

The original code execution output of IPython Kernel is also provided below for reference:

(Empty DataFrame
Columns: [content, _time_, _source_, _namespace_, _pod_uid_, _container_ip_, _image_name_, _container_name_, _pod_name_, __topic__, __tag__:__pack_id__, __tag__:_node_ip_, __tag__:_cluster_id_, __tag__:__hostname__, __tag__:_node_name_]
Index: []

[0 rows x 15 columns], Empty DataFrame
Columns: [content, _time_, _source_, _namespace_, _pod_uid_, _container_ip_, _image_name_, _container_name_, _pod_name_, __topic__, __tag__:__pack_id__, __tag__:_node_ip_, __tag__:_cluster_id_, __tag__:__hostname__, __tag__:_node_name_]
Index: []

[0 rows x 15 columns])```
```

In [ ]:
# In[7]:

# Load the topology.json file
topology_file = f'{task_directory}/topology.json'
with open(topology_file, 'r') as file:
    topology_data = json.load(file)

# Extract entities and edges
entities = pd.DataFrame(topology_data['entities'])
edges = pd.DataFrame(topology_data['edges'])

# Find the ID of the `product-catalog` service
product_catalog_entity = entities[
    (entities['type'] == 'apm.service') &
    (entities['name'].str.contains('product-catalog', case=False, na=False))
]

# If the service is found, identify its upstream and downstream services
if not product_catalog_entity.empty:
    product_catalog_id = product_catalog_entity.iloc[0]['id']
    upstream_services = edges[edges['dst'] == product_catalog_id]
    downstream_services = edges[edges['src'] == product_catalog_id]
else:
    product_catalog_id = None
    upstream_services = pd.DataFrame()
    downstream_services = pd.DataFrame()

product_catalog_entity, upstream_services, downstream_services

```
Out[7]:
```
The `product-catalog` service was identified in the topology with the ID `ae41c357daeed41f49e51340e779af71`. It has three upstream services calling it, but their specific names are not provided in the edges data. Additionally, the `product-catalog` service contains two operations (`455f921f0afc16175724d9f27faf18e4` and `1948ddf615871ac6fc18d01f72817e86`) and two instances (`df6348543e3360ea1206bd20179899fc` and `34e3e1c865aacce7dbda259b7ca3cefc`). These components are part of its internal structure.

The original code execution output of IPython Kernel is also provided below for reference:

(                                  id         type             name  first_observed  last_observed                                              props
15  ae41c357daeed41f49e51340e779af71  apm.service  product-catalog               0     1777001595  {'service': 'product-catalog', 'service_id': '...,                                   src     src_type                               dst     dst_type relation  first_observed  last_observed
92   55e5d48e4d1bdc8b07f350cb89476ad1  apm.service  ae41c357daeed41f49e51340e779af71  apm.service    calls               0     1775949133
93   7b50436b7fb9e828f0a8c32a69c53ab0  apm.service  ae41c357daeed41f49e51340e779af71  apm.service    calls               0     1775949133
118  70af2dee3d886988a1f1baefbf5fc400  apm.service  ae41c357daeed41f49e51340e779af71  apm.service    calls               0     1775949013,                                   src     src_type                               dst       dst_type  relation  first_observed  last_observed
65   ae41c357daeed41f49e51340e779af71  apm.service  455f921f0afc16175724d9f27faf18e4  apm.operation  contains               0     1775949133
83   ae41c357daeed41f49e51340e779af71  apm.service  df6348543e3360ea1206bd20179899fc   apm.instance  contains               0     1775949133
97   ae41c357daeed41f49e51340e779af71  apm.service  1948ddf615871ac6fc18d01f72817e86  apm.operation  contains               0     1775949133
108  ae41c357daeed41f49e51340e779af71  apm.service  34e3e1c865aacce7dbda259b7ca3cefc   apm.instance  contains               0     1775949133)```
```